# LES TRANSFORMERS : Résumé de texte

In [ ]:
# Importation et installation des packages
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import matplotlib.pyplot as plt
!pip install rouge_score
from rouge_score import rouge_scorer
!pip install bert_score
from bert_score import score
device = "cuda" if torch.cuda.is_available() else "cpu"


Tokenisation

In [ ]:
# Modèle BART
model_name = "moussaKam/barthez-orangesum-abstract"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, attn_implementation="eager").to(device)
model.eval()

# Pour récupérer les attentions quand tu feras le forward après génération
model.config.output_attentions = True
model.config.return_dict = True

print("Model:", model_name)


Model: moussaKam/barthez-orangesum-abstract


Le modele BARThez est un modele entraîné spécifiquement grace à OrangeSum,un jeu de données  français qui regroupe des articles d’actualité avec leurs résumés et parfois leurs titres.

In [ ]:
# Texte source à résumer ( il s'agit d'un extrait d'un article du Monde : https://www.lemonde.fr/societe/article/2025/12/11/l-etat-de-sante-des-francais-toujours-tres-marque-par-les-inegalites-sociales_6656941_3224.html)
source_text = (
    """
De la naissance à la fin de vie, les inégalités socio-économiques pèsent sur les parcours de soins. On ne compte plus les alertes, de chercheurs comme de médecins, sur ces écarts qui touchent à la santé de la population, entre ses franges les plus aisées et celles qui le sont moins. L’édition 2024 du baromètre de Santé publique France (SPF), rendue publique jeudi 11 décembre, permet d’en mesurer l’étendue.
Santé mentale, alcool, tabac, sommeil, sédentarité… Au total, 35 000 personnes, âgées de 18 à 79 ans, ont été interrogées, entre février et mai 2024, autour de 20 grands enjeux de santé publique, sur leurs habitudes de vie, leurs connaissances et la perception de leur état de santé. La photographie qui en ressort comporte des « nouvelles encourageantes » mais révèle aussi de « grands défis », ont fait valoir les porte-parole de SPF, jeudi.
« On observe l’existence d’inégalités socio-économiques de façon systématique pour l’ensemble des critères de santé étudiés », a rapporté, devant la presse, Stéphanie Vandentorren, épidémiologiste chez SPF. Dans le flot de statistiques diffusées, beaucoup l’attestent : si les personnes interrogées sont plus de deux sur trois (68 %) à déclarer une « bonne » ou une « très bonne » santé générale, le ratio atteint 82,5 % parmi les individus à l’aise financièrement, contre 50,4 % pour ceux qui déclarent une situation financière difficile.
Un changement de méthodologie dans la collecte des données, intervenu en 2024, ne permet pas de mesurer l’évolution dans le temps pour l’ensemble des paramètres. Reste que les inégalités en santé transparaissent, toujours, de manière criante. Tour d’horizon
"""
)

print("\n===== TEXTE SOURCE =====\n")
print(source_text)


===== TEXTE SOURCE =====


De la naissance à la fin de vie, les inégalités socio-économiques pèsent sur les parcours de soins. On ne compte plus les alertes, de chercheurs comme de médecins, sur ces écarts qui touchent à la santé de la population, entre ses franges les plus aisées et celles qui le sont moins. L’édition 2024 du baromètre de Santé publique France (SPF), rendue publique jeudi 11 décembre, permet d’en mesurer l’étendue.
Santé mentale, alcool, tabac, sommeil, sédentarité… Au total, 35 000 personnes, âgées de 18 à 79 ans, ont été interrogées, entre février et mai 2024, autour de 20 grands enjeux de santé publique, sur leurs habitudes de vie, leurs connaissances et la perception de leur état de santé. La photographie qui en ressort comporte des « nouvelles encourageantes » mais révèle aussi de « grands défis », ont fait valoir les porte-parole de SPF, jeudi.
« On observe l’existence d’inégalités socio-économiques de façon systématique pour l’ensemble des critères de santé ét

In [ ]:
# Tokenisation

inputs = tokenizer(
    source_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
).to(device)

input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

print("\nTOKENISATION")
print("input_ids shape:", tuple(input_ids.shape))
print("attention_mask shape:", tuple(attention_mask.shape))

src_tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
print("Extrait tokens:", src_tokens[:60])



TOKENISATION
input_ids shape: (1, 354)
attention_mask shape: (1, 354)
Extrait tokens: ['<s>', '▁De', '▁la', '▁naissance', '▁à', '▁la', '▁fin', '▁de', '▁vie', ',', '▁les', '▁inégalités', '▁socio', '-', 'économiques', '▁pèsent', '▁sur', '▁les', '▁parcours', '▁de', '▁soins', '.', '▁On', '▁ne', '▁compte', '▁plus', '▁les', '▁alertes', ',', '▁de', '▁chercheurs', '▁comme', '▁de', '▁médecins', ',', '▁sur', '▁ces', '▁écarts', '▁qui', '▁touchent', '▁à', '▁la', '▁santé', '▁de', '▁la', '▁population', ',', '▁entre', '▁ses', '▁fran', 'ges', '▁les', '▁plus', '▁ais', 'ées', '▁et', '▁celles', '▁qui', '▁le', '▁sont']


In [ ]:
#  Génération du résumé (Decodeur auto-régressif + softmax)

gen = model.generate(
    **inputs,
    num_beams=6,
    min_new_tokens=35,
    max_new_tokens=120,
    no_repeat_ngram_size=3,
    repetition_penalty=1.10,
    length_penalty=1.0,
    early_stopping=True,
    return_dict_in_generate=True
)

summary_ids = gen.sequences
generated_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("\n===== RÉSUMÉ GÉNÉRÉ =====\n")
print(generated_summary)



===== RÉSUMÉ GÉNÉRÉ =====

Si les Français sont plus de deux sur trois à déclarer une bonne » ou une « très bonne » santé générale, le ratio atteint 82,5 % parmi les individus à l’aise financièrement, contre 50,4 % pour ceux qui déclarent une situation financière difficile.


## On force le modèle à "rejouer" ce résumé pour pouvoir récupérer les attentions (encoder + cross-attention).

In [ ]:
decoder_input_ids = summary_ids.clone()

with torch.no_grad():
    outputs = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        decoder_input_ids=decoder_input_ids,
        output_attentions=True,
        return_dict=True
    )

encoder_attentions = outputs.encoder_attentions
cross_attentions = outputs.cross_attentions

print(f"\nNombre de couches encodeur : {len(encoder_attentions)}")
print(f"Nombre de couches cross-attention : {len(cross_attentions)}")


Nombre de couches encodeur : 6
Nombre de couches cross-attention : 6



## On récupère l'attention de la dernière couche de cross-attention.

In [ ]:
last_cross = cross_attentions[-1]
attn_matrix = last_cross[0, 0]

In [ ]:
# Affichage de la cross-attention

att_matrix = attn_matrix_clean.detach().cpu().numpy()

# summary_tokens : liste des tokens du résumé (lignes)
# source_tokens  : liste des tokens du texte source (colonnes)
df_att = pd.DataFrame(att_matrix,
                      index=tgt_tokens,
                      columns=src_tokens_clean)

# Pour éviter un tableau immense,affichons un extrait
df_att.round(3).iloc[:10,:10]

,L,Ã©,on,ard,Ġde,ĠVin,ci,Ġ(,ital,ien
</s>,0.097,0.011,0.003,0.001,0.001,0.001,0.001,0.010,0.006,0.001
<s>,0.097,0.011,0.003,0.001,0.001,0.001,0.001,0.010,0.006,0.001
L,0.018,0.263,0.034,0.002,0.002,0.005,0.001,0.001,0.000,0.000
Ã©,0.008,0.095,0.086,0.006,0.000,0.007,0.001,0.000,0.000,0.000
on,0.012,0.010,0.043,0.039,0.007,0.023,0.003,0.004,0.000,0.000
ard,0.123,0.004,0.009,0.019,0.152,0.066,0.002,0.050,0.001,0.000
Ġde,0.022,0.001,0.002,0.001,0.005,0.500,0.008,0.001,0.000,0.000
ĠVin,0.000,0.000,0.001,0.002,0.001,0.024,0.124,0.004,0.000,0.001
ci,0.008,0.001,0.000,0.000,0.001,0.001,0.000,0.085,0.002,0.000
Ġ(,0.018,0.001,0.000,0.000,0.000,0.000,0.000,0.004,0.171,0.002


In [ ]:
# Observons les tokens ayant une attention > 0.3
threshold = 0.3
mask = df_att > threshold

pairs_forts = []

for i, summary_tok in enumerate(df_att.index):
    for j, source_tok in enumerate(df_att.columns):
        val = df_att.iat[i, j]
        if val > threshold:
            pairs_forts.append((summary_tok, source_tok, float(val)))

# Trier par attention décroissante
pairs_forts = sorted(pairs_forts, key=lambda x: x[2], reverse=True)

for s_tok, src_tok, v in pairs_forts[:10]:
    print(f"{s_tok:15s}  -->  {src_tok:15s}  | attention = {v:.3f}")


ĠÃł              -->  ĠAm              | attention = 0.779
Ġser             -->  ĠPier            | attention = 0.757
Ġla              -->  ĠRenaissance     | attention = 0.706
acles            -->  Ġet              | attention = 0.693
ien              -->  Ġ:               | attention = 0.611
Ġda              -->  ĠVin             | attention = 0.597
,                -->  L                | attention = 0.571
Ġdi              -->  ĠPier            | attention = 0.523
Ġde              -->  ĠVin             | attention = 0.500
ateur            -->  Ġet              | attention = 0.463


| Symbole / Token | Signification |
|---|---|
| `▁` | Espace avant / début de mot |
| `Ġ` | Espace avant |
| `Ċ` | Retour à la ligne (`\n`) |
| `##` | Suite de mot (à coller au token précédent) |
| `<s>` | Début de séquence |
| `</s>` | Fin de séquence |
| `<pad>` | Padding (remplissage) |
| `<unk>` | Token inconnu |
| `<mask>` | Token masqué |


Au moment au le modèle génère le token ĠÃł dans le résumé, il regarde tres fortement le token ĠAm du texte source à hauteur de 77,9%

 Analyse d'une cross-attention:



## Calcul des scores ROUGE



### Notations
- R = référence
- C = résumé candidat
- G₁(X) = ensemble (avec répétitions) des tokens de X
- countₓ(g) = nombre d’occurrences du token g dans X

### Overlap₁
Overlap₁ = Σ_{g ∈ G₁(C)} min( count_C(g), count_R(g) )

### Recall (ROUGE-1)
ROUGE-1_recall = Overlap₁ / Σ_{g ∈ G₁(R)} count_R(g)

### Precision (ROUGE-1)
ROUGE-1_precision = Overlap₁ / Σ_{g ∈ G₁(C)} count_C(g)

### F1 (ROUGE-1)
ROUGE-1_F1 = 2 * (ROUGE-1_precision * ROUGE-1_recall) / (ROUGE-1_precision + ROUGE-1_recall)


### Recall (ROUGE-2)
ROUGE-2_recall = Overlap₂ / Σ_{g ∈ G₂(R)} count_R(g)

### Precision (ROUGE-2)
ROUGE-2_precision = Overlap₂ / Σ_{g ∈ G₂(C)} count_C(g)

### F1 (ROUGE-2)
ROUGE-2_F1 = 2 * (ROUGE-2_precision * ROUGE-2_recall) / (ROUGE-2_precision + ROUGE-2_recall)

ROUGE-L_recall = LCS(C, R) / |R|

### Precision (ROUGE-L)
ROUGE-L_precision = LCS(C, R) / |C|

### F1 (ROUGE-L)
ROUGE-L_F1 = 2 * (ROUGE-L_precision * ROUGE-L_recall) / (ROUGE-L_precision + ROUGE-L_recall)



**UTILITE :**

ROUGE-1 F1 : qualité globale du vocabulaire / des infos présentes.

ROUGE-2 F1 : capture mieux les enchaînements de mots → un peu plus strict.

ROUGE-L F1 : sensibilité à la structure de phrase, ordre des mots.

In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

scores = scorer.score( source_text, generated_summary)

print("\n===== Scores ROUGE (Hypothèse: Résumé généré, Référence: Texte source) =====\n")
for rouge_type, score_tuple in scores.items():
    print(f"{rouge_type.upper()}:")
    print(f"  F-measure: {score_tuple.fmeasure:.4f}")
    print(f"  Precision: {score_tuple.precision:.4f}")
    print(f"  Recall:    {score_tuple.recall:.4f}")
    print()



===== Scores ROUGE (Hypothèse: Résumé généré, Référence: Texte source) =====

ROUGE1:
  F-measure: 0.2701
  Precision: 0.9792
  Recall:    0.1567

ROUGE2:
  F-measure: 0.2486
  Precision: 0.9149
  Recall:    0.1438

ROUGEL:
  F-measure: 0.2644
  Precision: 0.9583
  Recall:    0.1533



ROUGE1 :  
- Recall :Le résummé prend en compte qu'une petite partie du vocabulaire du texte source

- Précision : présences des mots utilisé dans le texte résumé très élevé

- F Measure: Résumé court, mais ne couvre pas tout

ROUGE2:
- Recall :Peu d’enchaînements lexicaux du texte sont repris

- Les paires de mots générées existent majoritairement dans le texte source

ROUGEL:
- Recall: La structure globale du texte n’est que partiellement couverte


##  Calcul du Bertscore

BERTScore, c’est une métrique qui évalue la qualité d’un texte généré (résumé, traduction,...) en comparant son sens à un texte de référence grâce aux embeddings BERT, pas juste aux mots exacts.


## Formule du BERTScore

### Notations
- R = référence (résumé humain)
- C = résumé candidat (généré)
- rᵢ = embedding contextualisé du iᵉ token de R (issu de BERT)
- cⱼ = embedding contextualisé du jᵉ token de C
- cos(a, b) = similarité cosinus entre deux vecteurs

---

### Similarité token-token
sim(cⱼ, rᵢ) = cos(cⱼ, rᵢ)

---

### Precision (BERTScore-P)
Pour chaque token du résumé candidat, on garde la similarité maximale avec un token de la référence :

BERTScore_precision = (1 / |C|) × Σⱼ maxᵢ sim(cⱼ, rᵢ)

---

### Recall (BERTScore-R)
Pour chaque token de la référence, on garde la similarité maximale avec un token du résumé candidat :

BERTScore_recall = (1 / |R|) × Σᵢ maxⱼ sim(rᵢ, cⱼ)

---

### F1 (BERTScore-F1)
BERTScore_F1 = 2 × (Precision × Recall) / (Precision + Recall)


Plus ces scores sont proches de 1, plus notre texte généré est sémantiquement proche du texte de référence.

In [ ]:


# Résumé généré par notre modèle
candidate_summary = generated_summary

# Résumé humain de référence (à écrire par moi-même, ou pris d’un dataset)
reference_summary = (
    """Le baromètre 2024 de Santé publique France confirme que les inégalités socio-économiques impactent tous les indicateurs de santé (mental, alcool, tabac, sommeil, sédentarité…).
    Basé sur 35 000 personnes (18–79 ans) interrogées entre février et mai 2024, il montre un écart net : 82,5 % des plus à l’aise déclarent une bonne santé, contre 50,4 % des personnes en difficulté. Malgré un changement de méthode, le constat reste criant."""
)

# 3) BERTScore attend des listes
candidates = [candidate_summary]      # sortie du modèle
references = [reference_summary]      # gold standard humain

P, R, F1 = score(
    candidates,
    references,
    lang="fr",       # 'fr' pour français, 'en' pour anglais, etc.
    verbose=True
)

print("\n===== Scores BERTScore (Hypothèse: Résumé généré, Référence: Résumé humain) =====\n")
print(f"Precision: {P.mean().item():.4f}")
print(f"Recall:    {R.mean().item():.4f}")
print(f"F1-Score:  {F1.mean().item():.4f}")


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 2.07 seconds, 0.48 sentences/sec

===== Scores BERTScore (Hypothèse: Résumé généré, Référence: Résumé humain) =====

Precision: 0.7497
Recall:    0.6781
F1-Score:  0.7121


- Précision: les tokens du résumé généré sont proches à 74.97% de ceux du résumé humain

- Recall: Le résumé généré couvre une part significative des informations présentes dans le résumé humain, mais pas l’intégralité

- F1 Score : Bon compromis entre précision et couverture

In [ ]:
# Calcul du taux de compréssion du texte source

len_summary = len(generated_summary)
len_source = len(source_text)

if len_source > 0:
    compression_rate = len_summary / len_source
else:
    compression_rate = 0.0

print(f"Longueur du résumé: {len_summary} caractères")
print(f"Longueur du texte source: {len_source} caractères")
print(f"Taux de compression (longueur résumé / longueur texte source): {compression_rate:.4f}")


Longueur du résumé: 247 caractères
Longueur du texte source: 1653 caractères
Taux de compression (longueur résumé / longueur texte source): 0.1494


Le résumé ne conserve qu’environ 15 % de la longueur du texte initial.